<a href="https://colab.research.google.com/github/vinutasalagur017-cmyk/Cardiovascular-Disease-Prediction/blob/main/Spotify_Genre_Segmentation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Spotify Songs Genre Segmentation & Music Recommendation
> **Project Overview:** Using Unsupervised Machine Learning (K-Means Clustering & PCA) to segment Spotify songs based on audio features and build a content-based recommendation system.

---

## 📊 Project Description
This project analyzes a dataset of over **32,800 Spotify tracks** across 6 major playlist genres and 24 subgenres. By analyzing intrinsic audio features such as *Danceability, Energy, Valence, Acousticness, Tempo, and Loudness*, we group songs with similar acoustic properties together and build the foundation for an automated music recommendation system.

---

## 📂 Dataset Overview
* **Total Records:** 32,833 tracks
* **Genres:** 6 (`edm`, `rap`, `pop`, `r&b`, `latin`, `rock`)
* **Subgenres:** 24 subgenres
* **Key Audio Features:** Danceability, Energy, Key, Loudness, Mode, Speechiness, Acousticness, Instrumentalness, Liveness, Valence, Tempo, Duration

---

## 🛠️ Technologies & Libraries Used
* **Python**: Core programming language
* **Pandas & NumPy**: Data cleaning, transformation, and matrix computations
* **Scikit-learn**: StandardScaler, K-Means Clustering, PCA Dimensionality Reduction, Cosine Similarity
* **Matplotlib & Seaborn**: Comprehensive Exploratory Data Analysis and Cluster Visualizations

---

## 🚀 Machine Learning & Data Science Workflow
1. **Data Loading & Inspection**
2. **Data Cleaning & Missing Value Handling**
3. **Feature Engineering & Audio Feature Separation**
4. **Exploratory Data Analysis (EDA) & Visualizations**
5. **Correlation Analysis & Heatmap**
6. **Feature Standardization (StandardScaler)**
7. **Optimal Cluster Determination (Elbow Method & Silhouette Analysis)**
8. **K-Means Clustering & 2D PCA Visualization**
9. **Cluster Profiling & Genre / Subgenre / Playlist Cross-Analysis**
10. **Cluster Interpretation & Data-Driven Naming**
11. **Content-Based Music Recommendation System Demo**
12. **Conclusions & Key Findings**

---

## 📚 1. Library Imports and Environment Setup

In this section, we import the required Python libraries for data processing, statistical visualization, machine learning clustering, and recommendation similarity computation.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity

import warnings
warnings.filterwarnings('ignore')

# Visualization aesthetics
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
pd.set_option('display.max_columns', None)

RANDOM_STATE = 42
print('✅ All libraries imported successfully!')

## 📂 2. Dataset Loading and Initial Exploration

We load the Spotify songs dataset into a Pandas DataFrame. The loader automatically checks for common file locations (local path, Colab uploaded file, or default workspace CSV).

In [ ]:
# Locate and load dataset
possible_paths = [
    'spotify_songs.csv',
    'spotify_dataset.csv',
    'upload_1cc29b48-7982-4cdb-a0c8-c647c3cd2aa1.csv',
    'data/spotify_songs.csv',
    '../data/spotify_songs.csv'
]

data_path = None
for p in possible_paths:
    if os.path.exists(p):
        data_path = p
        break

if data_path is None:
    raise FileNotFoundError('Dataset file not found. Please upload spotify_songs.csv.')

df = pd.read_csv(data_path)
print(f'✅ Dataset loaded successfully from: {data_path}')
print(f'Dimensions: {df.shape[0]:,} rows × {df.shape[1]} columns')
df.head()

## 🔍 3. Dataset Structure and Statistical Summary

Let's inspect the data types, non-null counts, and summary statistics across all numerical columns.

In [ ]:
print('=== Dataset Information ===')
df.info()

In [ ]:
print('=== Statistical Summary of Numerical Features ===')
df.describe().round(3)

## 🧹 4. Data Cleaning & Missing Value Handling

We check for missing values and duplicate rows. In our dataset, only 5 rows contain missing strings in metadata (`track_name`, `track_artist`, `track_album_name`), which we cleanly remove.

In [ ]:
print('=== Missing Value Analysis ===')
missing_counts = df.isnull().sum()
print(missing_counts[missing_counts > 0])

# Drop the 5 missing rows
initial_len = len(df)
df = df.dropna(subset=['track_name', 'track_artist', 'track_album_name']).reset_index(drop=True)
print(f'\nRemoved {initial_len - len(df)} rows with missing values.')
print(f'Cleaned dataset size: {len(df):,} rows.')

# Duplicate row analysis
print(f'Exact duplicate rows: {df.duplicated().sum()}')
print(f'Unique track IDs: {df["track_id"].nunique():,} (songs in multiple playlists are preserved)')

## ⚙️ 5. Feature Engineering & Separation of Audio Features

We create two interpretable features:
1. `duration_minutes`: Track length in minutes (from `duration_ms`).
2. `release_year`: Extracted 4-digit release year from `track_album_release_date`.

We also separate **Metadata columns** (identifiers, track names) from the **12 Numerical Audio Features** used for clustering.

In [ ]:
# Feature Engineering
df['duration_minutes'] = df['duration_ms'] / 60000.0
df['release_year'] = df['track_album_release_date'].astype(str).str[:4].astype(int)

# Numerical Audio Features for Clustering
# These represent the sonic characteristics of tracks (tempo, rhythm, timbre, loudness)
audio_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness',
    'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo',
    'duration_minutes'
]

# Metadata columns (kept for reporting and recommendation display)
metadata_columns = [
    'track_id', 'track_name', 'track_artist', 'track_popularity',
    'track_album_id', 'track_album_name', 'track_album_release_date',
    'playlist_name', 'playlist_id', 'playlist_genre', 'playlist_subgenre'
]

print(f'✅ Audio features selected for clustering ({len(audio_features)}):')
print(audio_features)

## 📊 6. Exploratory Data Analysis (EDA)

We create detailed, high-quality visualizations to analyze genre proportions, subgenres, popularity distributions, and acoustic properties.

### 6.1 Genre Distribution in Dataset

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

genre_counts = df['playlist_genre'].value_counts()
colors = sns.color_palette('Set2', len(genre_counts))

# Bar chart
bars = axes[0].bar(genre_counts.index, genre_counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[0].set_title('Distribution of Playlist Genres', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Genre', fontsize=12)
axes[0].set_ylabel('Number of Songs', fontsize=12)
for bar, val in zip(bars, genre_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 60, f'{val:,}', ha='center', va='bottom', fontweight='bold', fontsize=10)

# Pie chart
axes[1].pie(genre_counts.values, labels=genre_counts.index, autopct='%1.1f%%', colors=colors, startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Genre Proportions', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 6.2 Top 12 Playlist Subgenres

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
subgenre_counts = df['playlist_subgenre'].value_counts().head(12)
bars = ax.barh(subgenre_counts.index[::-1], subgenre_counts.values[::-1], color=sns.color_palette('viridis', len(subgenre_counts)), edgecolor='black', linewidth=0.4)
ax.set_title('Top 12 Playlist Subgenres', fontsize=14, fontweight='bold')
ax.set_xlabel('Number of Songs', fontsize=12)
ax.set_ylabel('Subgenre', fontsize=12)
for bar, val in zip(bars, subgenre_counts.values[::-1]):
    ax.text(bar.get_width() + 15, bar.get_y() + bar.get_height()/2, f'{val:,}', ha='left', va='center', fontsize=10)
plt.tight_layout()
plt.show()

### 6.3 Track Popularity Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Popularity histogram
axes[0].hist(df['track_popularity'], bins=50, color='steelblue', edgecolor='black', linewidth=0.4)
axes[0].set_title('Distribution of Track Popularity', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Popularity Score (0-100)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].axvline(df['track_popularity'].mean(), color='red', linestyle='--', label=f'Mean: {df["track_popularity"].mean():.1f}')
axes[0].legend()

# Popularity boxplot by genre
sns.boxplot(data=df, x='playlist_genre', y='track_popularity', palette='Set2', ax=axes[1])
axes[1].set_title('Track Popularity by Genre', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Genre', fontsize=11)
axes[1].set_ylabel('Popularity Score', fontsize=11)

plt.tight_layout()
plt.show()

### 6.4 Audio Feature Distributions

In [ ]:
dist_features = ['danceability', 'energy', 'valence', 'acousticness', 'instrumentalness', 'speechiness', 'loudness', 'tempo']

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for i, feat in enumerate(dist_features):
    axes[i].hist(df[feat], bins=40, color=sns.color_palette('tab10')[i], edgecolor='black', linewidth=0.3, alpha=0.8)
    axes[i].set_title(feat.capitalize(), fontsize=12, fontweight='bold')
    axes[i].set_xlabel(feat, fontsize=10)
    axes[i].set_ylabel('Frequency', fontsize=10)
    axes[i].axvline(df[feat].mean(), color='black', linestyle='--', label=f'Mean: {df[feat].mean():.2f}')
    axes[i].legend(fontsize=9)

plt.suptitle('Audio Feature Distributions Across Spotify Songs', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

### 6.5 Audio Features Compared Across Genres

In [ ]:
comp_feats = ['danceability', 'energy', 'valence', 'acousticness', 'speechiness', 'loudness']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, feat in enumerate(comp_feats):
    sns.boxplot(data=df, x='playlist_genre', y=feat, palette='Set2', ax=axes[i])
    axes[i].set_title(f'{feat.capitalize()} by Genre', fontsize=12, fontweight='bold')
    axes[i].set_xlabel('Genre', fontsize=10)
    axes[i].set_ylabel(feat.capitalize(), fontsize=10)

plt.suptitle('Audio Feature Comparison Across Playlist Genres', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 🔗 7. Correlation Analysis & Feature Heatmap

We calculate the Pearson correlation matrix across numerical audio features to discover meaningful acoustic associations.

In [ ]:
corr_cols = ['track_popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_minutes']
corr_matrix = df[corr_cols].corr().round(2)

fig, ax = plt.subplots(figsize=(14, 9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, cmap='RdBu_r', center=0, fmt='.2f', linewidths=0.5, ax=ax, vmin=-1, vmax=1, annot_kws={'size': 10})
ax.set_title('Correlation Matrix of Audio Features', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Notable Correlation Insights:')
print('1. Energy & Loudness show a strong positive correlation (+0.68) - louder tracks exhibit higher perceived acoustic energy.')
print('2. Energy & Acousticness show a strong negative correlation (-0.54) - acoustic recordings tend to have softer, lower-energy profiles.')
print('3. Danceability & Valence show a moderate positive correlation (+0.33) - happier sounding tracks are more danceable.')

## ⚖️ 8. Feature Standardization (StandardScaler)

K-Means clustering uses Euclidean distance, which is sensitive to feature magnitudes. We standardize all 12 audio features to have $\mu = 0$ and $\sigma = 1$.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[audio_features])

print(f'Scaled feature matrix shape: {X_scaled.shape}')
print(f'Mean across features (approx 0): {np.abs(X_scaled.mean(axis=0)).max():.4e}')
print(f'Std dev across features (approx 1): {X_scaled.std(axis=0).round(4)}')

## 🎯 9. Determining the Optimal Number of Clusters

We evaluate cluster counts $k \in [2, 10]$ using the **Elbow Method (Inertia)** and **Silhouette Analysis**.

In [ ]:
k_range = range(2, 11)
inertias = []
silhouette_scores = []

print('Evaluating k values...')
for k in k_range:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    # Sample silhouette for fast evaluation
    score = silhouette_score(X_scaled, labels, sample_size=10000, random_state=RANDOM_STATE)
    silhouette_scores.append(score)
    print(f'  k={k}: Inertia = {km.inertia_:.2f}, Silhouette Score = {score:.4f}')

# Plot Elbow and Silhouette curves side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Elbow plot
axes[0].plot(list(k_range), inertias, 'bo-', linewidth=2, markersize=7)
axes[0].set_title('Elbow Method for Optimal k', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=11)
axes[0].set_xticks(list(k_range))

# Silhouette plot
axes[1].plot(list(k_range), silhouette_scores, 'rs-', linewidth=2, markersize=7)
axes[1].axvline(6, color='green', linestyle='--', label='Selected k=6')
axes[1].set_title('Silhouette Score vs Number of Clusters', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[1].set_ylabel('Silhouette Score', fontsize=11)
axes[1].set_xticks(list(k_range))
axes[1].legend()

plt.tight_layout()
plt.show()

print('\n✅ Cluster Count Decision: We select k = 6 as it captures distinct acoustic segments and naturally aligns with the 6 foundational Spotify playlist genres.')

## 🔬 10. Final K-Means Clustering Model

We fit the final K-Means model with $k = 6$ and assign each song to its audio cluster.

In [ ]:
optimal_k = 6
kmeans_model = KMeans(n_clusters=optimal_k, random_state=RANDOM_STATE, n_init=10)
df['cluster'] = kmeans_model.fit_predict(X_scaled)

print(f'✅ K-Means clustering completed with k={optimal_k}!')
print(f'Final Model Inertia: {kmeans_model.inertia_:.2f}')
print('\nSongs per cluster:')
print(df['cluster'].value_counts().sort_index())

## 📉 11. 2D Dimensionality Reduction via PCA

We apply Principal Component Analysis (PCA) to project the 12-dimensional audio feature space into 2 dimensions for visual inspection of the clusters.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_scaled)
df['pca_1'] = X_pca[:, 0]
df['pca_2'] = X_pca[:, 1]

var_1 = pca.explained_variance_ratio_[0] * 100
var_2 = pca.explained_variance_ratio_[1] * 100
total_var = sum(pca.explained_variance_ratio_) * 100

print(f'PCA Variance Explained: PC1 = {var_1:.2f}%, PC2 = {var_2:.2f}%, Total = {total_var:.2f}%')

# 2D Scatter Plot
fig, ax = plt.subplots(figsize=(12, 7))
scatter = ax.scatter(df['pca_1'], df['pca_2'], c=df['cluster'], cmap='viridis', alpha=0.35, s=9)
cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Cluster ID', fontsize=11)
ax.set_title('2D PCA Projection of Song Clusters (k=6)', fontsize=14, fontweight='bold')
ax.set_xlabel(f'Principal Component 1 ({var_1:.2f}% Variance)', fontsize=12)
ax.set_ylabel(f'Principal Component 2 ({var_2:.2f}% Variance)', fontsize=12)
plt.tight_layout()
plt.show()

## 📊 12. Cluster Profiling & Cross-Analysis

We analyze the audio profiles of each cluster and cross-tabulate clusters against **Playlist Genres**, **Subgenres**, and **Curated Playlists**.

### 12.1 Average Audio Feature Profiles by Cluster

In [ ]:
cluster_profiles = df.groupby('cluster')[audio_features].mean().round(3)
print('=== Average Audio Feature Means per Cluster ===')
print(cluster_profiles)

# Normalized heatmap
profiles_norm = (cluster_profiles - cluster_profiles.min()) / (cluster_profiles.max() - cluster_profiles.min())
fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(profiles_norm.T, annot=cluster_profiles.T.values, cmap='YlGnBu', fmt='', linewidths=0.5, ax=ax)
ax.set_title('Audio Feature Profiles per Cluster (Normalized Color, Actual Means Annotated)', fontsize=13, fontweight='bold')
ax.set_xlabel('Cluster ID', fontsize=11)
ax.set_ylabel('Audio Feature', fontsize=11)
plt.tight_layout()
plt.show()

### 12.2 Genre vs Cluster Cross-Tabulation

In [ ]:
genre_crosstab = pd.crosstab(df['playlist_genre'], df['cluster'])

fig, axes = plt.subplots(1, 2, figsize=(18, 6))
# Count heatmap
sns.heatmap(genre_crosstab, annot=True, fmt='d', cmap='Blues', linewidths=0.5, ax=axes[0])
axes[0].set_title('Genre vs Cluster (Track Count)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Cluster', fontsize=11)
axes[0].set_ylabel('Genre', fontsize=11)

# Percentage heatmap (row-normalized)
genre_pct = genre_crosstab.div(genre_crosstab.sum(axis=1), axis=0) * 100
sns.heatmap(genre_pct, annot=True, fmt='.1f', cmap='YlGnBu', linewidths=0.5, ax=axes[1])
axes[1].set_title('Genre vs Cluster (% of Genre)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Cluster', fontsize=11)
axes[1].set_ylabel('Genre', fontsize=11)

plt.tight_layout()
plt.show()

### 12.3 Top 10 Curated Playlists across Clusters

In [ ]:
top_playlists = df['playlist_name'].value_counts().head(10).index
df_top_pl = df[df['playlist_name'].isin(top_playlists)]
pl_crosstab = pd.crosstab(df_top_pl['playlist_name'], df_top_pl['cluster'])

fig, ax = plt.subplots(figsize=(14, 7))
pl_crosstab.plot(kind='barh', stacked=True, colormap='viridis', ax=ax, edgecolor='black', linewidth=0.3)
ax.set_title('Cluster Distribution for Top 10 Curated Spotify Playlists', fontsize=13, fontweight='bold')
ax.set_xlabel('Track Count', fontsize=11)
ax.set_ylabel('Playlist Name', fontsize=11)
ax.legend(title='Cluster', bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 🏷️ 13. Data-Driven Cluster Interpretation

Based on calculated means and dominant genres, we assign descriptive interpretations to each cluster.

In [ ]:
print('=' * 85)
print('SUMMARY OF DISCOVERED CLUSTERS')
print('=' * 85)

cluster_names = {
    0: 'Danceable & Energetic (Minor Key) - Dominant: Latin / Pop',
    1: 'Upbeat Dance & Pop (Major Key) - Dominant: Latin / Pop',
    2: 'High-Energy Rock & Fast-Tempo EDM - Dominant: Rock (32.4%), EDM (29.8%)',
    3: 'Acoustic, Mellow & Downtempo - Dominant: R&B (33.0%), Pop',
    4: 'Electronic & Instrumental Focused - Dominant: EDM (58.5%)',
    5: 'Speech-Heavy & Rhythmic Rap - Dominant: Rap (54.4%)'
}

for c_id in sorted(df['cluster'].unique()):
    c_data = df[df['cluster'] == c_id]
    dom_genre = c_data['playlist_genre'].mode().iloc[0]
    dom_sub = c_data['playlist_subgenre'].mode().iloc[0]
    print(f'\n▶ Cluster {c_id}: {cluster_names[c_id]}')
    print(f'   Total Songs: {len(c_data):,}')
    print(f'   Dominant Genre: {dom_genre} | Dominant Subgenre: {dom_sub}')
    print(f'   Key Means: Dance={c_data["danceability"].mean():.2f}, Energy={c_data["energy"].mean():.2f}, Acoustic={c_data["acousticness"].mean():.2f}, Instrumental={c_data["instrumentalness"].mean():.2f}, Speech={c_data["speechiness"].mean():.2f}')

print('=' * 85)

## 🎵 14. Content-Based Song Recommendation Engine

We build a content-based recommendation system that calculates the **Cosine Similarity** between audio vectors: 
$$\text{Similarity}(\vec{u}, \vec{v}) = \frac{\vec{u} \cdot \vec{v}}{\|\vec{u}\|_2 \|\vec{v}\|_2}$$
A small bonus ($+0.05$) is added for songs belonging to the same cluster to enhance genre and sonic coherence.

In [ ]:
def recommend_songs(track_name, df, X_scaled, n_recs=5, cluster_boost=0.05):
    """
    Recommend similar songs using cosine similarity on scaled audio features.
    """
    matches = df[df['track_name'].str.contains(track_name, case=False, na=False)]
    if matches.empty:
        print(f'❌ Song "{track_name}" not found in dataset.')
        return pd.DataFrame()
    
    # Select match with highest popularity
    query_song = matches.sort_values(by='track_popularity', ascending=False).iloc[0]
    query_idx = query_song.name
    query_vec = X_scaled[query_idx].reshape(1, -1)
    
    # Compute cosine similarities
    sims = cosine_similarity(query_vec, X_scaled).flatten()
    
    # Add same-cluster boost
    cluster_mask = (df['cluster'] == query_song['cluster']).values.astype(float) * cluster_boost
    final_scores = np.round(sims + cluster_mask, 4)
    
    results = df.copy()
    results['similarity_score'] = final_scores
    results = results[results.index != query_idx]
    results = results.drop_duplicates(subset=['track_name'], keep='first')
    results = results.sort_values(by='similarity_score', ascending=False).head(n_recs)
    
    cols = ['track_name', 'track_artist', 'playlist_genre', 'playlist_subgenre', 'similarity_score', 'cluster']
    
    print(f'🎧 Query Track: "{query_song["track_name"]}" by {query_song["track_artist"]}')
    print(f'   Genre: {query_song["playlist_genre"]} | Cluster: {query_song["cluster"]}')
    print('-' * 75)
    return results[cols]

print('✅ Recommendation engine defined successfully!')

### 14.1 Recommendation Demonstration on Sample Tracks

In [ ]:
# Test 1: Popular Track
test_1 = df.nlargest(10, 'track_popularity')['track_name'].iloc[0]
recs_1 = recommend_songs(test_1, df, X_scaled, n_recs=5)
display(recs_1)

# Test 2: Rock / EDM Track
test_2 = 'Memories - Dillon Francis Remix'
recs_2 = recommend_songs(test_2, df, X_scaled, n_recs=5)
display(recs_2)

## 🎓 15. Conclusions & Key Findings

### Summary of Results
1. **Preprocessing & Audio Features:** Standardizing 12 key audio features allows unsupervised segmentation across multi-dimensional musical properties without bias from differing feature scales.
2. **Acoustic Correlation Insights:** Energy strongly correlates with Loudness (+0.68) and negatively with Acousticness (-0.54), reflecting standard acoustic production characteristics.
3. **Clustering Performance ($k=6$):** The K-Means model segments tracks into interpretable groups (e.g. *Instrumental EDM, Speech-Heavy Rap, Acoustic/Mellow, Upbeat Pop*).
4. **Recommendation Effectiveness:** Vector cosine similarity on standardized audio features successfully identifies sonically similar tracks, providing a cold-start-free content-based music recommender.

---
*Developed as an academic minor project for Spotify Songs Genre Segmentation and Recommendation.*